# Land registry paper: experiments in Google Colab

Run the cells from top to bottom. Everything is saved in your Google Drive folder, so nothing is lost if Colab disconnects.

**Before you start**, create the folder `land_paper` in **My Drive** with this content:
```
land_paper/
  code.zip
  gv/gv_village_summary.csv
  gv/gv_real_sample.csv
  data/Base.csv          (BAF, from Kaggle)
  data/creditcard.csv    (Credit Card, from Kaggle)
```

## Step 0: Connect Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/land_paper'
%cd {PROJECT}
!unzip -o -q code.zip
!mkdir -p data results figures
!ls

In [ ]:
!pip install -q xgboost lightgbm tabulate
import xgboost, lightgbm, sklearn
print('ok', sklearn.__version__, xgboost.__version__, lightgbm.__version__)

## Step 1: Generate the land dataset

First list the taluks in your guideline-value file. Choose the ones in **Madurai district** (the zone also includes Dindigul and Theni) and type them into `TALUKS` in the next cell.

In [ ]:
import pandas as pd
gv = pd.read_csv('gv/gv_village_summary.csv')
print(gv.groupby('taluk').village.nunique().sort_values(ascending=False).to_string())

In [ ]:
# Replace these with the Madurai taluk names printed above (exact spelling).
# Leave the list empty to use the whole zone (then write "Madurai registration zone" in the paper).
TALUKS = ['Madurai North', 'Melur', 'Vadipatti']

taluk_args = ' '.join(f'"{t}"' for t in TALUKS)
!python code/generate_land_data.py --gv gv/gv_village_summary.csv {'--taluks ' + taluk_args if TALUKS else ''} --n 80000 --out data/land_transactions.csv

Copy the report printed above into Section 3.2 and Table 2. It is also saved as `data/land_transactions_report.md`.

## Step 2: Fidelity check (Table 4)

In [ ]:
!python code/fidelity_check.py --synthetic data/land_transactions.csv --real gv/gv_real_sample.csv --columns market_value_per_acre --group village
!python code/fidelity_check.py --synthetic data/land_transactions.csv --fraud-type-col fraud_type --audit-counts undervaluation=151 misclassification=47

## Step 3: Quick test on 15,000 rows (about 30 min)

`fraud_type` is always dropped. It is the answer key, and keeping it would make every model score 100%.

In [ ]:
DROP = 'transaction_id seller_id buyer_id survey_number fraud_type'
pd.read_csv('data/land_transactions.csv').sample(15000, random_state=1).to_csv('data/land_small.csv', index=False)
!python code/run_experiments.py --data data/land_small.csv --target is_fraud --drop {DROP} --out results/test --skip-5x2
print(open('results/test/tables.md').read()[:3000])

## Step 4: Full run on 80,000 rows (Tables 5, 7, 8, 9)

This takes several hours. Keep the browser tab open; free Colab disconnects after about 90 minutes without interaction and stops after about 12 hours.
If it stops partway, run the cell again. To save time, run it once with `--skip-5x2` first, and add the 5×2cv run later.

In [ ]:
!python code/run_experiments.py --data data/land_transactions.csv --target is_fraud --drop {DROP} --out results/land

## Step 5: Public benchmarks (Tables 3, 6, 8)

In [ ]:
# BAF: stratified 20% sample (200,000 rows)
d = pd.read_csv('data/Base.csv')
d.groupby('fraud_bool', group_keys=False).apply(lambda g: g.sample(frac=0.2, random_state=42)).to_csv('data/baf_200k.csv', index=False)
del d
!python code/run_experiments.py --data data/baf_200k.csv --target fraud_bool --date-col none --out results/baf --skip-5x2

In [ ]:
!python code/run_experiments.py --data data/creditcard.csv --target Class --date-col none --out results/cc --skip-5x2

## Step 6: Slither (Table 12)

Upload your contract to `land_paper/contracts/MaduraiLandRegistry.sol` first. Set the Solidity version to match the `pragma` line in your file.

In [ ]:
!pip install -q slither-analyzer solc-select
SOLC = '0.8.20'   # match your pragma
!solc-select install {SOLC} && solc-select use {SOLC}
!slither contracts/MaduraiLandRegistry.sol --json results/slither.json || true
import json, collections
r = json.load(open('results/slither.json'))
print(collections.Counter(x['impact'] for x in r['results'].get('detectors', [])))

## Step 7: Blockchain measurements

Run `bench_blockchain.js` **on your laptop** with Ganache and IPFS (see the implementation guide). Colab is not suitable for latency measurements.
Then upload `blockchain_results.json` to `land_paper/` and run Step 8.

## Step 8: Figures (Figs 2–4)

In [ ]:
!python code/make_figures.py --results results/land --blockchain blockchain_results.json --out figures
!ls figures

## Step 9: Download everything

In [ ]:
!cd {PROJECT} && zip -q -r results_for_paper.zip results figures data/land_transactions_report.md
from google.colab import files
files.download(f'{PROJECT}/results_for_paper.zip')

Send `results_for_paper.zip` to Claude to fill the tables and write the results text.